In [ ]:
# ── Cell 1: Mount Google Drive and set working directory ──────────────────
from google.colab import drive # type: ignore # type: ignore
import os

drive.mount("/content/drive")

# ⚠️  Update this path to wherever you put the project folder on your Drive.
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/PixelLab-StudyPal-RAG-DIP/smart-learning-assistant"

os.chdir(DRIVE_PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir("."))

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
# RAGAS 0.1.x + Groq as judge LLM (free, no billing required)
%pip install -q \
    "ragas==0.1.21" \
    "datasets>=2.18" \
    "langchain>=1.2" \
    "langchain-groq" \
    "langchain-community>=0.4" \
    "langchain-huggingface" \
    "sentence-transformers" \
    "python-dotenv"

print("\n✅ Packages installed.")


In [ ]:
# ── Cell 3: Set Groq API key (free judge LLM for RAGAS) ───────────────────
# Get a free key at https://console.groq.com/keys (sign in with GitHub/Google)
import os
from getpass import getpass

groq_key = getpass("Enter your Groq API key (gsk_...): ")
os.environ["GROQ_API_KEY"] = groq_key

# Quick sanity check — uses llama-3.1-8b-instant (500K TPD, much lower token cost)
from langchain_groq import ChatGroq  # type: ignore
_test = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_key, temperature=0)
_resp = _test.invoke("Reply with the single word OK.")
print(f"✅ Groq key valid. Test response: {_resp.content.strip()}")
print("   RAGAS will use: llama-3.1-8b-instant (judge, 500K TPD) + all-MiniLM-L6-v2 (embeddings)")
print("   RunConfig: max_workers=2, timeout=120s  →  no timeout cascade")


In [ ]:
# ── Cell 4: Load intermediate JSON ────────────────────────────────────────
import json
from pathlib import Path

INTERMEDIATE_PATH = Path("data/eval_intermediate.json")

if not INTERMEDIATE_PATH.exists():
    # Fallback: allow manual upload from local machine
    from google.colab import files # type: ignore
    print("eval_intermediate.json not found in Drive project folder.")
    print("Uploading from your local machine instead...")
    uploaded = files.upload()          # shows a file-picker in Colab
    INTERMEDIATE_PATH = Path(list(uploaded.keys())[0])

with open(INTERMEDIATE_PATH, encoding="utf-8") as f:
    intermediate = json.load(f)

n_questions  = len(intermediate["questions"])
n_guardrail  = len(intermediate.get("guardrail_results", []))
collected_at = intermediate.get("collected_at", "unknown")

print(f"✅ Loaded intermediate data")
print(f"   DIP questions  : {n_questions}")
print(f"   Guardrail tests: {n_guardrail}")
print(f"   Collected at   : {collected_at}")

# Preview first question
print(f"\n   Q1: {intermediate['questions'][0]}")
print(f"   A1: {intermediate['answers'][0][:120]}...")

In [ ]:
# ── Cell 5: Run RAGAS scoring + generate report ───────────────────────────
import sys
sys.path.insert(0, str(Path.cwd()))

from app.evaluation.metrics import run_ragas_scoring, generate_report

print("Running RAGAS evaluation (using reduced concurrency to avoid timeouts)...")
ragas_df = run_ragas_scoring(
    INTERMEDIATE_PATH,
    max_workers=2,   # 2 parallel jobs → avoids Groq TPM burst / timeout cascade
    timeout=120,     # 2 min per job ceiling
)

# Pretty-print scores
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
targets = {"faithfulness": 0.7, "answer_relevancy": 0.7, "context_precision": 0.7, "context_recall": 0.7}

print("\n=== RAGAS Scores ===")
print(f"{'Metric':<25} {'Score':>7}  {'Target':>7}  Status")
print("-" * 52)
for m in metrics:
    if m in ragas_df.columns:
        score  = ragas_df[m].mean()
        target = targets[m]
        status = "✅ PASS" if score >= target else "❌ FAIL"
        print(f"{m:<25} {score:>7.3f}  {target:>7.1f}  {status}")

print("\nGenerating markdown report...")
report_md = generate_report(
    ragas_df,
    latencies=intermediate.get("latencies", []),
    guardrail_results=intermediate.get("guardrail_results", []),
    topic_map=intermediate.get("topic_map"),
)
print("✅ Report generated.")

In [ ]:
# ── Cell 6: Save report to Drive and offer local download ─────────────────
from pathlib import Path
from google.colab import files as colab_files # type: ignore

REPORT_PATH = Path("evaluation_report.md")

# The report was already written to evaluation_report.md by generate_report()
if REPORT_PATH.exists():
    print(f"📄 Report saved at: {REPORT_PATH.resolve()}")
    print(f"   Size: {REPORT_PATH.stat().st_size:,} bytes")

    # Download to local machine
    print("\nDownloading evaluation_report.md to your local machine...")
    colab_files.download(str(REPORT_PATH))

    # Also save RAGAS DataFrame as CSV
    csv_path = Path("data/ragas_scores.csv")
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    ragas_df.to_csv(csv_path, index=False)
    colab_files.download(str(csv_path))
    print("✅ Downloaded evaluation_report.md and ragas_scores.csv")
else:
    print("⚠️  Report file not found. Check for errors in Cell 5.")

# Print the report inline too
print("\n" + "=" * 70)
print(report_md)